# Nss Midcourse project: Creating pathfinder 2e monster encounter
### combine all input datasheets to create each character build

In [1]:
#Import libraries 
import pandas as pd
import numpy as np
import requests
import matplotlib as plt


import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### The structure of a character sheet:
* ability scores: strength, dexterity, constitution, intelligence, wisdonm, charisma
    * ability modifiers= 8-9 = -1, 10-11 = 0, 12-13= +1, 14=15= +2, 16-17 = +3, 18-19= +4
* proficency:
  * untrained= +0
  * trained= 2+ level
  * expert= 4+ level
  * master = 6+ level
  * legendary = 8+ level
* Defenses: armor class, fortitude, reflex, will, hit points (HP + con)
* Equipment proficencies: armor proficencies, weapon proficencies (unarmored, light, medium, heavy)
* perception: wisdom + proficency
* Skills: Acrobatics, Arcana, Athletics, Crafting, Deception, Diplomacy, Intimidation, Medicine, Nature, Religion, Occultism, Performance, Society, Stealth, Survival, Thievery
      * simplify by removing skills
* feats
* spellcasting: magical tradiiton, spell attack, spell dc
* Character creation rules: up to 4 free boosts, ability cannot exceed 18 at level 1

### At character creation (base): 
* all ability scores are 10
* Everything is untrained

### Create a dataframe with all of these features as a row, and input the values depending on the build combination

In [2]:
character_sheet_rows= ['strength','dexterity', 'constitution', 'intelligence', 'wisdom', 'charisma', 'armor class', 
                       'fortitude', 'reflex', 'will', 'hit points', 'perception', 'speed', 'unarmored', 'light', 'medium', 'heavy', 
                       'unarmed', 'simple', 'martial', 'advanced', 'other',
                       'magical tradition', 'spell attack', 'spell dc']

character_sheet= pd.DataFrame(character_sheet_rows, columns=['build metrics'])
character_sheet= character_sheet.set_index('build metrics', drop=True)
character_sheet

#add a column for each level
base_formula= ['10+mod','10+mod', '10+mod', '10+mod', '10+mod', '10+mod','10+dex+prof+item','10+con+prof+item', 
                'dex+prof+item', 'wis+prof+item', 'class + ancestry + con', 'wis+prof', 'ancestry',  
              'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 
              'class', 'mod+prof', '10+mod+prof']

character_sheet['formula']= base_formula
character_sheet['ability_modifier']= ['str','dex', 'con', 'int', 'wis', 'char','dex', 'con', 
                'dex', 'wis', 'con', 'wis', 'ancestry',  
              'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 
              'class', 'class', 'class']
character_sheet['proficiency']= ['none','none', 'none', 'none', 'none', 'none','untrained','untrained', 
                'untrained', 'untrained', 'none', 'untrained', 'none', 'untrained', 
              'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained',
              'none', 'untrained', 'untrained']
character_sheet['base_value']= ['10','10', '10', '10', '10', '10','10','10', 
                '0', '0', '0', '0', '0', '0', 
              '0', '0', '0', '0', '0', '0', '0', '0',
              '0', '0', '0']

#add in filler rows for the information from class and ancestry choices
character_sheet['ancestry_boost']= character_sheet.index
character_sheet['class_boost']= character_sheet.index
character_sheet['equipment_boost']= character_sheet.index
character_sheet['bcakground_boost']= character_sheet.index
character_sheet['feat_boost']= character_sheet.index
character_sheet['free_boost']= character_sheet.index

character_sheet['level 1']= character_sheet.index
#character_sheet['level 2']= character_sheet.index
#character_sheet['level 3']= character_sheet.index
#character_sheet['level 4']= character_sheet.index
#character_sheet['level 5']= character_sheet.index
#character_sheet['level 6']= character_sheet.index
#character_sheet['level 7']= character_sheet.index
#character_sheet['level 8']= character_sheet.index
#character_sheet['level 9']= character_sheet.index
#character_sheet['level 10']= character_sheet.index


#save character sheet in base templates 
character_sheet.to_csv('../templates/character_sheet_template.csv')

#view dataframe
character_sheet

,formula,ability_modifier,proficiency,base_value,ancestry_boost,class_boost,equipment_boost,bcakground_boost,feat_boost,free_boost,level 1
build metrics,,,,,,,,,,,
strength,10+mod,str,none,10,strength,strength,strength,strength,strength,strength,strength
dexterity,10+mod,dex,none,10,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity
constitution,10+mod,con,none,10,constitution,constitution,constitution,constitution,constitution,constitution,constitution
intelligence,10+mod,int,none,10,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence
wisdom,10+mod,wis,none,10,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom
charisma,10+mod,char,none,10,charisma,charisma,charisma,charisma,charisma,charisma,charisma
armor class,10+dex+prof+item,dex,untrained,10,armor class,armor class,armor class,armor class,armor class,armor class,armor class
fortitude,10+con+prof+item,con,untrained,10,fortitude,fortitude,fortitude,fortitude,fortitude,fortitude,fortitude
reflex,dex+prof+item,dex,untrained,0,reflex,reflex,reflex,reflex,reflex,reflex,reflex


In [3]:
# character_sheet['ability_modifier']= ['str','dex', 'con', 'int', 'wis', 'char','dex', 'dex', 'con', 
#                 'dex', 'wis', 'con', 'wis', 'ancestry', 'class', 
#               'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 
#               'dex', 'int', 'str', 'int', 'cha', 'cha', 'cha', 
#               'wis', 'wis', 'wis', 'int', 'cha', 'int', 'dex', 'wis', 'dex',
#               'class', 'class']

# character_sheet= pd.DataFrame(character_sheet_rows, columns=['build metrics'])
# character_sheet= character_sheet.set_index('build metrics', drop=True)
# character_sheet

# #add a column for each level
# base_formula= ['10+mod','10+mod', '10+mod', '10+mod', '10+mod', '10+mod','10+dex+prof+item','10+con+prof+item', 
#                 'dex+prof+item', 'wis+prof+item', 'class + con', 'wis+prof', 'ancestry', '10+mod+prof', 
#               'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 'prof', 
#               'dex+prof', 'int+prof', 'str+prof', 'int+prof', 'cha+prof', 'cha+prof', 'cha+prof', 
#               'wis+prof', 'wis+prof', 'wis+prof', 'int+prof', 'cha+prof', 'int+prof', 'dex+prof', 'wis+prof', 'dex+prof',
#               'magical tradiiton', 'mod+prof', '10+mod+prof']

# character_sheet['formula']= base_formula
# character_sheet['ability_modifier']= ['str','dex', 'con', 'int', 'wis', 'char','dex', 'dex', 'con', 
#                 'dex', 'wis', 'con', 'wis', 'ancestry', 'class', 
#               'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 
#               'dex', 'int', 'str', 'int', 'cha', 'cha', 'cha', 
#               'wis', 'wis', 'wis', 'int', 'cha', 'int', 'dex', 'wis', 'dex',
#               'class', 'class']
# character_sheet['proficiency']= ['none','none', 'none', 'none', 'none', 'none','untrained','untrained', 
#                 'untrained', 'untrained', 'none', 'untrained', 'none', 'untrained', 
#               'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 
#               'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 
#               'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained', 'untrained',
#               'none', 'untrained', 'untrained']
# character_sheet['base_value']= ['10','10', '10', '10', '10', '10','10','10', 
#                 '0', '0', '0', '0', '0', '0', 
#               '0', '0', '0', '0', '0', '0', '0', '0', '0', 
#               '0', '0', '0', '0', '0', '0', '0', 
#               '0', '0', '0', '0', '0', '0', '0', '0', '0',
#               '0', '0', '0']

# #add in filler rows for the information from class and ancestry choices
# character_sheet['ancestry_boost']= character_sheet.index
# character_sheet['class_boost']= character_sheet.index
# character_sheet['equipment_boost']= character_sheet.index
# character_sheet['feat_boost']= character_sheet.index

# character_sheet['level 1']= character_sheet.index
# character_sheet['level 2']= character_sheet.index
# character_sheet['level 3']= character_sheet.index
# character_sheet['level 4']= character_sheet.index
# character_sheet['level 5']= character_sheet.index
# character_sheet['level 6']= character_sheet.index
# character_sheet['level 7']= character_sheet.index
# character_sheet['level 8']= character_sheet.index
# character_sheet['level 9']= character_sheet.index
# character_sheet['level 10']= character_sheet.index


# #save character sheet in base templates 
# character_sheet.to_csv('../templates/character_sheet_template.csv')

# #view dataframe
# character_sheet

### Open ancestry attribute stats

In [4]:
ancestry_stat= pd.read_csv('../output_data/ancestry_attribute.csv')
ancestry_stat= ancestry_stat.drop('Unnamed: 0', axis=1)
ancestry_stat= ancestry_stat.set_index('Attribute', drop=True)
ancestry_stat.loc['Speed']= [x.split(' ')[0] for x in ancestry_stat.loc['Speed']]
ancestry_stat.loc['Speed'] = ancestry_stat.loc['Speed'].astype(int) #25- ancestry_stat.loc['Speed'].astype(int)

### add two different human and orc builds by chosing the base ability boosts to streamline 

ancestry_stat['human-str'] = ancestry_stat['human']

ancestry_stat['human-dex'] = ancestry_stat['human']
ancestry_stat['orc-str'] = ancestry_stat['orc']
ancestry_stat['orc-dex'] = ancestry_stat['orc']

#loop between the new columns to add in the ability boosts to the main stat AND constitution 
free_ability=['Strength', 'Strength', 'Constitution', 'Constitution','Dexterity', 'Dexterity', 'Constitution', 'Constitution']
free_ability_class=['human-str', 'orc-str','human-str', 'orc-str', 'human-dex', 'orc-dex', 'human-dex', 'orc-dex']

for row in range (len(free_ability_class)):
    ancestry_stat.at[free_ability[row], free_ability_class[row]] = '+2'

#drop the free boost column and the normal human and orc columns
ancestry_stat= ancestry_stat.drop(['Two free ability boosts'], axis=0)	
ancestry_stat= ancestry_stat.drop(['human', 'orc'], axis=1)	

#save character sheet in base templates 
ancestry_stat.to_csv('../templates/ancestry_stat_boost_template.csv')

#view dataframe 
ancestry_stat


,dwarf,elf,halfling,human-str,human-dex,orc-str,orc-dex
Attribute,,,,,,,
Hit Points,10,6,6,8,8,10,10
Speed,20,30,25,25,25,25,25
Strength,0,0,-2,+2,0,+2,0
Dexterity,0,+2,+2,0,+2,0,+2
Constitution,+2,-2,0,+2,+2,+2,+2
Intelligence,0,+2,0,0,0,0,0
Wisdom,+2,0,+2,0,0,0,0
Charisma,-2,0,0,0,0,0,0


### Open class stats

In [5]:
class_base_stat= pd.read_csv('../output_data/all_class_base_stats.csv')
class_base_stat= class_base_stat.set_index('Class', drop=True).drop(['Unnamed: 0'], axis=1)	
class_base_stat

#clean up primary ability column for easier future filtering
class_base_stat['Primary Ability']= [x.split('[')[-1] for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']= [x.split(']')[0] for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']
#Use explode to create new rows based on primary ability scores
class_base_stat['Primary Ability']= [re.sub(r'\[.*?\]', '', x) for x in class_base_stat['Primary Ability']] 
class_base_stat['Primary Ability']= [x.split(',') for x in class_base_stat['Primary Ability']]
class_base_stat= class_base_stat.explode('Primary Ability')

##add extra columns for filtering later
class_base_stat['ability boost'] = '+2'
class_base_stat['class_build']= class_base_stat.index + ' '+class_base_stat['Primary Ability'].str.replace("'", '')

#save spreadsheet
class_base_stat

,Primary Ability,Hit Points per Level,Perception,Fortitude,Reflex,Will,Skills,Defenses,Attacks,Spells,Class/Spell DC,ability boost,class_build
Class,,,,,,,,,,,,,
Alchemist,'Intelligence',8,Trained,Expert,Expert,Trained,"['Crafting', '+3 of Choice']","Light, Unarmored","Simple, Unarmed, Alchemical Bombs",--,Trained,+2,Alchemist Intelligence
Barbarian,'Strength',12,Expert,Expert,Trained,Expert,"['Athletics', '+3 of Choice']","Light, Medium, Unarmored","Simple, Martial, Unarmed",--,Trained,+2,Barbarian Strength
Bard,'Charisma',8,Expert,Trained,Trained,Expert,"['Occultism and Performance', '+ 4 of Choice']","Light, Unarmored","Simple, Unarmed, Longsword, Rapier, Sap, Short...",Occult,Trained,+2,Bard Charisma
Champion,'Strength ',10,Trained,Expert,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","All, Unarmored","Simple, Martial, Unarmed",Divine,Trained,+2,Champion Strength
Champion,' Dexterity',10,Trained,Expert,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","All, Unarmored","Simple, Martial, Unarmed",Divine,Trained,+2,Champion Dexterity
Cleric,'Wisdom',8,Trained,Trained,Trained,Expert,"[""Religion and 1 of your Deity's"", '+2 of Choi...","Unarmored, Armor noted by Doctrine","Simple, Deity's Favored Weapon, Unarmed",Divine,Trained,+2,Cleric Wisdom
Druid,'Wisdom',8,Trained,Trained,Trained,Expert,"['Nature and 1 from your Order', '+2 of Choice']","Light, Medium, Unarmored","Simple, Unarmed",Primal,Trained,+2,Druid Wisdom
Fighter,'Strength ',10,Expert,Expert,Expert,Trained,"['Acrobatics OR Athletics', '+3 of Choice']","All, Unarmored","Advanced; EXPERT: Simple, Martial, Unarmed",--,Trained,+2,Fighter Strength
Fighter,' Dexterity',10,Expert,Expert,Expert,Trained,"['Acrobatics OR Athletics', '+3 of Choice']","All, Unarmored","Advanced; EXPERT: Simple, Martial, Unarmed",--,Trained,+2,Fighter Dexterity


In [6]:
character_sheet

,formula,ability_modifier,proficiency,base_value,ancestry_boost,class_boost,equipment_boost,bcakground_boost,feat_boost,free_boost,level 1
build metrics,,,,,,,,,,,
strength,10+mod,str,none,10,strength,strength,strength,strength,strength,strength,strength
dexterity,10+mod,dex,none,10,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity
constitution,10+mod,con,none,10,constitution,constitution,constitution,constitution,constitution,constitution,constitution
intelligence,10+mod,int,none,10,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence
wisdom,10+mod,wis,none,10,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom
charisma,10+mod,char,none,10,charisma,charisma,charisma,charisma,charisma,charisma,charisma
armor class,10+dex+prof+item,dex,untrained,10,armor class,armor class,armor class,armor class,armor class,armor class,armor class
fortitude,10+con+prof+item,con,untrained,10,fortitude,fortitude,fortitude,fortitude,fortitude,fortitude,fortitude
reflex,dex+prof+item,dex,untrained,0,reflex,reflex,reflex,reflex,reflex,reflex,reflex


In [7]:
#make a list of ancestry-class combinations?
all_ancestry= ancestry_stat.columns.to_list()
all_class_list= class_base_stat['class_build'].to_list()

chose_class_list=['Barbarian Strength', 'Fighter Strength ', 'Fighter   Dexterity', 'Rogue Dexterity ', 'Sorcerer Charisma']

#loop between each character combination to make a unique character sheet, and store it as a new data frame 
ancestry_stat['attribute_merge']= ancestry_stat.index.str.lower()
ancestry_stat= ancestry_stat.set_index('attribute_merge')

#make a loop
ancestry_choice = all_ancestry[0]
class_choice= chose_class_list[0]
build_name= ancestry_choice +' ' + class_choice
build_name

'dwarf Barbarian Strength'

In [8]:
####
# Ancestry stat boosts
####
#convert chosen ANCESTRY stats to a dictionary 
ancestry_value= ancestry_stat[f'{ancestry_choice}'].to_dict()

#replace the ancestry_boost values with the dictionary 
character_sheet['ancestry_boost'] = character_sheet['ancestry_boost'].replace(ancestry_value)
#fill all non-matching values with 0

####
#CLass stat boosts
####

#convert chosen class stats to a dictionary
class_merge= class_base_stat[class_base_stat['class_build'] == chose_class_list[0]].T
class_merge['metrics']= class_merge.index.str.lower()
class_merge= class_merge.set_index('metrics', drop=True)
class_merge= class_merge.rename(index={'hit points per level': 'hit points'})
class_value= class_merge.to_dict()

#replace the class_boost values with the class dictionary 
character_sheet['class_boost']= character_sheet['class_boost'].replace(class_value[class_merge.columns[0]])
character_sheet

#add the +2 for the class ability boost
class_ability_boost= class_value[class_merge.columns[0]]['primary ability'].lower().replace("'", "").strip()
character_sheet['class_boost']= character_sheet['class_boost'].replace(class_ability_boost, '+2')

character_sheet

####
#Class trained armor and weapon proficencies 
####
#pull out the trained levels for the armor and defense categories from the class dataframe 
trained_defense= class_merge.loc['defenses'].str.split(",")[-1]
trained_weapon= class_merge.loc['attacks'].str.split(",")[-1]

## loop through the string of trained armor and defenses on the class info spreadsheet
trained_weapon_defense=[]
for x in trained_weapon:
    new_string= x.lower().strip() #.replace("'", '')
    trained_weapon_defense.append(new_string)
for x in trained_defense:
    new_string= x.lower().strip()
    trained_weapon_defense.append(new_string)

#change the value in the class_boost to trained
trained_weapon_sheet=[]
for metric in character_sheet['class_boost']:
    if metric in trained_weapon_defense:
        trained_weapon_sheet.append('trained')
    else:
        trained_weapon_sheet.append(np.nan)

#update the character sheet with the armor and weapon defenses 
character_sheet['proficiency_class']= trained_weapon_sheet
character_sheet['proficiency']= character_sheet['proficiency_class'].fillna(character_sheet['class_boost'])
character_sheet= character_sheet.drop('proficiency_class', axis=1)
character_sheet

character_sheet

,formula,ability_modifier,proficiency,base_value,ancestry_boost,class_boost,equipment_boost,bcakground_boost,feat_boost,free_boost,level 1
build metrics,,,,,,,,,,,
strength,10+mod,str,+2,10,0,+2,strength,strength,strength,strength,strength
dexterity,10+mod,dex,dexterity,10,0,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity
constitution,10+mod,con,constitution,10,+2,constitution,constitution,constitution,constitution,constitution,constitution
intelligence,10+mod,int,intelligence,10,0,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence
wisdom,10+mod,wis,wisdom,10,+2,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom
charisma,10+mod,char,charisma,10,-2,charisma,charisma,charisma,charisma,charisma,charisma
armor class,10+dex+prof+item,dex,armor class,10,armor class,armor class,armor class,armor class,armor class,armor class,armor class
fortitude,10+con+prof+item,con,Expert,10,fortitude,Expert,fortitude,fortitude,fortitude,fortitude,fortitude
reflex,dex+prof+item,dex,Trained,0,reflex,Trained,reflex,reflex,reflex,reflex,reflex


In [9]:
####
#Class trained armor and weapon proficencies 
####
#pull out the trained levels for the armor and defense categories from the class dataframe 
trained_defense= class_merge.loc['defenses'].str.split(",")[-1]
trained_weapon= class_merge.loc['attacks'].str.split(",")[-1]

## loop through the string of trained armor and defenses on the class info spreadsheet
trained_weapon_defense=[]
for x in trained_weapon:
    new_string= x.lower().strip() #.replace("'", '')
    trained_weapon_defense.append(new_string)
for x in trained_defense:
    new_string= x.lower().strip()
    trained_weapon_defense.append(new_string)

#change the value in the class_boost to trained
trained_weapon_sheet=[]
for metric in character_sheet['class_boost']:
    if metric in trained_weapon_defense:
        trained_weapon_sheet.append('trained')
    else:
        trained_weapon_sheet.append(np.nan)

#update the character sheet with the armor and weapon defenses 
character_sheet['proficiency_class']= trained_weapon_sheet
character_sheet['proficiency']= character_sheet['proficiency_class'].fillna(character_sheet['class_boost'])
character_sheet= character_sheet.drop('proficiency_class', axis=1)
character_sheet

character_sheet

,formula,ability_modifier,proficiency,base_value,ancestry_boost,class_boost,equipment_boost,bcakground_boost,feat_boost,free_boost,level 1
build metrics,,,,,,,,,,,
strength,10+mod,str,+2,10,0,+2,strength,strength,strength,strength,strength
dexterity,10+mod,dex,dexterity,10,0,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity
constitution,10+mod,con,constitution,10,+2,constitution,constitution,constitution,constitution,constitution,constitution
intelligence,10+mod,int,intelligence,10,0,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence
wisdom,10+mod,wis,wisdom,10,+2,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom
charisma,10+mod,char,charisma,10,-2,charisma,charisma,charisma,charisma,charisma,charisma
armor class,10+dex+prof+item,dex,armor class,10,armor class,armor class,armor class,armor class,armor class,armor class,armor class
fortitude,10+con+prof+item,con,Expert,10,fortitude,Expert,fortitude,fortitude,fortitude,fortitude,fortitude
reflex,dex+prof+item,dex,Trained,0,reflex,Trained,reflex,reflex,reflex,reflex,reflex


In [10]:
character_sheet['class_boost']= character_sheet['class_boost'].replace(class_ability_boost, '+2')
character_sheet

,formula,ability_modifier,proficiency,base_value,ancestry_boost,class_boost,equipment_boost,bcakground_boost,feat_boost,free_boost,level 1
build metrics,,,,,,,,,,,
strength,10+mod,str,+2,10,0,+2,strength,strength,strength,strength,strength
dexterity,10+mod,dex,dexterity,10,0,dexterity,dexterity,dexterity,dexterity,dexterity,dexterity
constitution,10+mod,con,constitution,10,+2,constitution,constitution,constitution,constitution,constitution,constitution
intelligence,10+mod,int,intelligence,10,0,intelligence,intelligence,intelligence,intelligence,intelligence,intelligence
wisdom,10+mod,wis,wisdom,10,+2,wisdom,wisdom,wisdom,wisdom,wisdom,wisdom
charisma,10+mod,char,charisma,10,-2,charisma,charisma,charisma,charisma,charisma,charisma
armor class,10+dex+prof+item,dex,armor class,10,armor class,armor class,armor class,armor class,armor class,armor class,armor class
fortitude,10+con+prof+item,con,Expert,10,fortitude,Expert,fortitude,fortitude,fortitude,fortitude,fortitude
reflex,dex+prof+item,dex,Trained,0,reflex,Trained,reflex,reflex,reflex,reflex,reflex


In [11]:
class_ability_boost= class_value[class_merge.columns[0]]['primary ability'].lower().replace("'", "").strip()
class_ability_boost

'strength'